# 22. Robotics action policies — ACT and Diffusion Policy

ACT and Diffusion Policy are kept here as the action-chunking / diffusion-policy references. π0 and FAST live in notebook 23 so each implementation remains readable instead of being compressed into one oversized notebook.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(12)
device = torch.device("cpu")


## 1. Shared ResNet-18 feature backbone


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, group_norm=False):
        super().__init__()

        def norm(channels):
            if not group_norm:
                return nn.BatchNorm2d(channels)
            groups = min(8, channels)
            while channels % groups:
                groups -= 1
            return nn.GroupNorm(groups, channels)

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.norm1 = norm(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.norm2 = norm(out_ch)
        if stride == 1 and in_ch == out_ch:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                norm(out_ch),
            )

    def forward(self, x):
        h = F.relu(self.norm1(self.conv1(x)))
        h = self.norm2(self.conv2(h))
        return F.relu(h + self.skip(x))


class ResNet18Features(nn.Module):
    def __init__(self, out_ch=32, group_norm=False):
        super().__init__()
        widths = [8, 12, 16, 24]
        if group_norm:
            stem_norm = nn.GroupNorm(8, widths[0])
        else:
            stem_norm = nn.BatchNorm2d(widths[0])
        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], 7, stride=2, padding=3, bias=False),
            stem_norm,
            nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        stages = []
        in_ch = widths[0]
        for index, width in enumerate(widths):
            stride = 1 if index == 0 else 2
            stages.append(
                nn.Sequential(
                    BasicBlock(in_ch, width, stride, group_norm),
                    BasicBlock(width, width, 1, group_norm),
                )
            )
            in_ch = width
        self.stages = nn.ModuleList(stages)
        self.proj = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, image):
        x = self.stem(image)
        for stage in self.stages:
            x = stage(x)
        return self.proj(x)


## 2. ACT — posterior padding mask, DETR 2D positions, 4/4/7 Transformer depths


In [ ]:
def detr_position_2d(batch, height, width, dim, device, temperature=10000):
    assert dim % 4 == 0
    mask = torch.zeros(batch, height, width, dtype=torch.bool, device=device)
    not_mask = ~mask
    y = not_mask.cumsum(1, dtype=torch.float32)
    x = not_mask.cumsum(2, dtype=torch.float32)
    eps = 1e-6
    y = y / (y[:, -1:, :] + eps) * (2 * math.pi)
    x = x / (x[:, :, -1:] + eps) * (2 * math.pi)
    half = dim // 2
    dim_t = torch.arange(half, dtype=torch.float32, device=device)
    dim_t = temperature ** (2 * torch.div(dim_t, 2, rounding_mode="floor") / half)
    pos_x = x[:, :, :, None] / dim_t
    pos_y = y[:, :, :, None] / dim_t
    pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), -1).flatten(-2)
    pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), -1).flatten(-2)
    return torch.cat((pos_y, pos_x), -1).flatten(1, 2)


def fixed_position_1d(length, dim, device):
    position = torch.arange(length, device=device, dtype=torch.float32)[:, None]
    freq = torch.exp(
        torch.arange(0, dim, 2, device=device, dtype=torch.float32)
        * (-math.log(10000.0) / dim)
    )
    out = torch.zeros(length, dim, device=device)
    out[:, 0::2] = torch.sin(position * freq)
    out[:, 1::2] = torch.cos(position * freq)
    return out


class ACT(nn.Module):
    def __init__(self, action_dim=3, dim=32, chunk=4, latent_dim=8):
        super().__init__()
        self.chunk = chunk
        self.latent_dim = latent_dim
        self.backbone = ResNet18Features(dim)
        self.posterior_cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.qpos_post = nn.Linear(action_dim, dim)
        self.action_post = nn.Linear(action_dim, dim)
        post_layer = nn.TransformerEncoderLayer(dim, 8, 2 * dim, batch_first=True)
        self.posterior = nn.TransformerEncoder(post_layer, 4)
        self.mu = nn.Linear(dim, latent_dim)
        self.logvar = nn.Linear(dim, latent_dim)
        self.latent_proj = nn.Linear(latent_dim, dim)
        self.qpos_proj = nn.Linear(action_dim, dim)
        self.additional_position = nn.Embedding(2, dim)
        obs_layer = nn.TransformerEncoderLayer(dim, 8, 2 * dim, batch_first=True)
        self.encoder = nn.TransformerEncoder(obs_layer, 4)
        dec_layer = nn.TransformerDecoderLayer(dim, 8, 2 * dim, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, 7)
        self.queries = nn.Parameter(torch.randn(1, chunk, dim) * 0.02)
        self.head = nn.Linear(dim, action_dim)
        self.is_pad_head = nn.Linear(dim, 1)

    def encode_posterior(self, qpos, actions, is_pad):
        batch = qpos.size(0)
        tokens = torch.cat(
            [
                self.posterior_cls.expand(batch, -1, -1),
                self.qpos_post(qpos).unsqueeze(1),
                self.action_post(actions),
            ],
            1,
        )
        position = fixed_position_1d(tokens.size(1), tokens.size(2), tokens.device)[None]
        prefix_pad = torch.zeros(batch, 2, dtype=torch.bool, device=tokens.device)
        padding = torch.cat([prefix_pad, is_pad], 1)
        hidden = self.posterior(tokens + position, src_key_padding_mask=padding)[:, 0]
        mu, logvar = self.mu(hidden), self.logvar(hidden)
        latent = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return latent, mu, logvar

    def forward(self, image, qpos, actions=None, is_pad=None):
        batch = image.size(0)
        if actions is None:
            latent = torch.zeros(batch, self.latent_dim, device=image.device)
            mu = logvar = torch.zeros_like(latent)
        else:
            latent, mu, logvar = self.encode_posterior(qpos, actions, is_pad)
        feature = self.backbone(image)
        _, _, height, width = feature.shape
        image_tokens = feature.flatten(2).transpose(1, 2)
        image_pos = detr_position_2d(batch, height, width, image_tokens.size(-1), image.device)
        memory = torch.cat(
            [
                self.latent_proj(latent).unsqueeze(1),
                self.qpos_proj(qpos).unsqueeze(1),
                image_tokens,
            ],
            1,
        )
        special_pos = self.additional_position.weight[None].expand(batch, -1, -1)
        memory = self.encoder(memory + torch.cat([special_pos, image_pos], 1))
        queries = self.queries.expand(batch, -1, -1)
        decoded = self.decoder(queries, memory)
        predicted_actions = self.head(decoded)
        predicted_padding = self.is_pad_head(decoded).squeeze(-1)
        return predicted_actions, predicted_padding, mu, logvar


def act_loss(prediction, target, is_pad, mu, logvar, kl_weight=10.0):
    l1 = F.l1_loss(prediction, target, reduction="none")
    l1 = (l1 * (~is_pad)[..., None]).mean()
    total_kld = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(-1).mean()
    return l1 + kl_weight * total_kld


act = ACT()
assert [len(stage) for stage in act.backbone.stages] == [2, 2, 2, 2]
assert len(act.posterior.layers) == 4
assert len(act.encoder.layers) == 4
assert len(act.decoder.layers) == 7
act_image = torch.randn(2, 3, 32, 32)
act_qpos = torch.randn(2, 3)
act_target = torch.randn(2, 4, 3)
act_pad = torch.tensor([[False, False, False, True], [False, False, True, True]])
act_pred, act_pad_logits, mu, logvar = act(
    act_image,
    act_qpos,
    act_target,
    act_pad,
)
assert act_pad_logits.shape == act_pad.shape
act_loss(act_pred, act_target, act_pad, mu, logvar).backward()


## 3. Diffusion Policy — ConditionalUnet1D and 100-step DDPM schedule


In [ ]:
def diffusion_embedding(timestep, dim):
    half = dim // 2
    freq = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=timestep.device).float()
        / max(half - 1, 1)
    )
    angle = timestep.float()[:, None] * freq[None]
    return torch.cat([angle.sin(), angle.cos()], -1)


class Conv1dBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5):
        super().__init__()
        groups = min(8, out_ch)
        while out_ch % groups:
            groups -= 1
        self.conv = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size,
            padding=kernel_size // 2,
        )
        self.norm = nn.GroupNorm(groups, out_ch)

    def forward(self, x):
        return F.mish(self.norm(self.conv(x)))


class CondRes1D(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, kernel_size=5):
        super().__init__()
        self.block1 = Conv1dBlock(in_ch, out_ch, kernel_size)
        self.block2 = Conv1dBlock(out_ch, out_ch, kernel_size)
        self.cond = nn.Sequential(
            nn.Mish(),
            nn.Linear(cond_dim, 2 * out_ch),
        )
        self.skip = (
            nn.Identity()
            if in_ch == out_ch
            else nn.Conv1d(in_ch, out_ch, 1)
        )
        self.out_ch = out_ch

    def forward(self, x, condition):
        hidden = self.block1(x)
        scale, bias = self.cond(condition).view(
            -1,
            2,
            self.out_ch,
            1,
        ).unbind(1)
        hidden = hidden * scale + bias
        hidden = self.block2(hidden)
        return hidden + self.skip(x)


class Downsample1d(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            stride=2,
            padding=1,
        )

    def forward(self, x):
        return self.conv(x)


class Upsample1d(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.ConvTranspose1d(
            channels,
            channels,
            kernel_size=4,
            stride=2,
            padding=1,
        )

    def forward(self, x):
        return self.conv(x)


class ConditionalUnet1D(nn.Module):
    def __init__(
        self,
        action_dim=3,
        cond_dim=32,
        dims=(16, 32, 64),
        kernel_size=5,
    ):
        super().__init__()
        self.cond_dim = cond_dim
        full_cond_dim = cond_dim + cond_dim
        all_dims = [action_dim, *dims]
        in_out = list(zip(all_dims[:-1], all_dims[1:]))

        self.down_modules = nn.ModuleList()
        for stage_index, (dim_in, dim_out) in enumerate(in_out):
            is_last = stage_index == len(in_out) - 1
            self.down_modules.append(
                nn.ModuleList(
                    [
                        CondRes1D(
                            dim_in,
                            dim_out,
                            full_cond_dim,
                            kernel_size,
                        ),
                        CondRes1D(
                            dim_out,
                            dim_out,
                            full_cond_dim,
                            kernel_size,
                        ),
                        nn.Identity()
                        if is_last
                        else Downsample1d(dim_out),
                    ]
                )
            )

        mid_dim = dims[-1]
        self.mid_modules = nn.ModuleList(
            [
                CondRes1D(
                    mid_dim,
                    mid_dim,
                    full_cond_dim,
                    kernel_size,
                ),
                CondRes1D(
                    mid_dim,
                    mid_dim,
                    full_cond_dim,
                    kernel_size,
                ),
            ]
        )

        self.up_modules = nn.ModuleList()
        reversed_stages = list(reversed(in_out[1:]))
        for stage_index, (dim_in, dim_out) in enumerate(reversed_stages):
            is_last = stage_index >= len(in_out) - 1
            self.up_modules.append(
                nn.ModuleList(
                    [
                        CondRes1D(
                            dim_out * 2,
                            dim_in,
                            full_cond_dim,
                            kernel_size,
                        ),
                        CondRes1D(
                            dim_in,
                            dim_in,
                            full_cond_dim,
                            kernel_size,
                        ),
                        nn.Identity()
                        if is_last
                        else Upsample1d(dim_in),
                    ]
                )
            )

        start_dim = dims[0]
        self.final = nn.Sequential(
            Conv1dBlock(start_dim, start_dim, kernel_size),
            nn.Conv1d(start_dim, action_dim, 1),
        )

    def forward(self, actions, timestep, observation_condition):
        time = diffusion_embedding(timestep, self.cond_dim)
        condition = torch.cat([time, observation_condition], -1)
        hidden = actions.transpose(1, 2)
        skips = []

        for block1, block2, downsample in self.down_modules:
            hidden = block1(hidden, condition)
            hidden = block2(hidden, condition)
            skips.append(hidden)
            hidden = downsample(hidden)

        for block in self.mid_modules:
            hidden = block(hidden, condition)

        for block1, block2, upsample in self.up_modules:
            skip = skips.pop()
            hidden = torch.cat([hidden, skip], dim=1)
            hidden = block1(hidden, condition)
            hidden = block2(hidden, condition)
            hidden = upsample(hidden)

        return self.final(hidden).transpose(1, 2)


class DiffusionPolicy(nn.Module):
    def __init__(self, action_dim=3, cond_dim=32):
        super().__init__()
        self.observation_encoder = ResNet18Features(
            cond_dim,
            group_norm=True,
        )
        self.noise_predictor = ConditionalUnet1D(
            action_dim=action_dim,
            cond_dim=cond_dim,
            kernel_size=5,
        )

    def encode_observation(self, image):
        feature_map = self.observation_encoder(image)
        return feature_map.mean(dim=(-2, -1))

    def forward(self, actions, timestep, image):
        condition = self.encode_observation(image)
        return self.noise_predictor(actions, timestep, condition)


def cosine_betas(steps=100):
    def alpha_bar(time):
        angle = (time + 0.008) / 1.008 * math.pi / 2
        return math.cos(angle) ** 2

    values = []
    for step in range(steps):
        t1 = step / steps
        t2 = (step + 1) / steps
        beta = 1 - alpha_bar(t2) / alpha_bar(t1)
        values.append(min(beta, 0.999))
    return torch.tensor(values)


@torch.no_grad()
def ddpm_reverse_step(predicted_noise, timestep, sample, betas):
    alphas = 1.0 - betas
    alpha_cumprod = torch.cumprod(alphas, dim=0)
    alpha_t = alphas[timestep]
    alpha_bar_t = alpha_cumprod[timestep]
    if timestep > 0:
        alpha_bar_previous = alpha_cumprod[timestep - 1]
    else:
        alpha_bar_previous = torch.tensor(1.0, device=sample.device)

    predicted_x0 = (
        sample
        - torch.sqrt(1 - alpha_bar_t) * predicted_noise
    ) / torch.sqrt(alpha_bar_t)
    mean = (
        torch.sqrt(alpha_bar_previous) * betas[timestep]
        / (1 - alpha_bar_t)
        * predicted_x0
        + torch.sqrt(alpha_t)
        * (1 - alpha_bar_previous)
        / (1 - alpha_bar_t)
        * sample
    )

    if timestep == 0:
        return mean

    variance = (
        betas[timestep]
        * (1 - alpha_bar_previous)
        / (1 - alpha_bar_t)
    )
    return mean + variance.sqrt() * torch.randn_like(sample)


@torch.no_grad()
def sample_diffusion_policy(model, image, horizon=8, action_dim=3):
    betas = cosine_betas(100).to(image.device)
    sample = torch.randn(
        image.size(0),
        horizon,
        action_dim,
        device=image.device,
    )

    for timestep_value in reversed(range(100)):
        timestep = torch.full(
            (image.size(0),),
            timestep_value,
            dtype=torch.long,
            device=image.device,
        )
        predicted_noise = model(sample, timestep, image)
        sample = ddpm_reverse_step(
            predicted_noise,
            timestep_value,
            sample,
            betas,
        )
    return sample


policy = DiffusionPolicy()
assert [len(stage) for stage in policy.noise_predictor.down_modules] == [3, 3, 3]
assert len(policy.noise_predictor.mid_modules) == 2
assert [len(stage) for stage in policy.noise_predictor.up_modules] == [3, 3]
assert any(
    isinstance(module, nn.GroupNorm)
    for module in policy.observation_encoder.modules()
)
assert len(cosine_betas(100)) == 100

policy_image = torch.randn(1, 3, 32, 32)
action_input = torch.randn(1, 8, 3)
prediction = policy(
    action_input,
    torch.tensor([50]),
    policy_image,
)
prediction.square().mean().backward()

sampled_actions = sample_diffusion_policy(
    policy,
    policy_image,
    horizon=8,
    action_dim=3,
)
assert sampled_actions.shape == action_input.shape


## Audit result

ACT keeps the padding-aware posterior/loss, learned special positions, action and padding heads, and 4/4/7 Transformer depths. Diffusion Policy keeps the GroupNorm ResNet-18 observation encoder, learned down/up sampling, the released ConditionalUnet1D stage topology, FiLM scale/bias conditioning, and the complete 100-step stochastic DDPM reverse path.
